In [1]:
# Standard libraries
import os
import inspect

# Data manipulation
import numpy as np
import pandas as pd

# Statistical functions
from scipy.stats import norm

# Models
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.semi_supervised import SelfTrainingClassifier

# Model selection
from sklearn.model_selection import train_test_split, GridSearchCV

# Evaluation metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    auc,
    confusion_matrix,
    classification_report,
    roc_curve,
    precision_recall_curve,
    log_loss,
)

# Visualization
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches

In [2]:
# SETRED
import sys
sys.path.append(os.path.abspath(".."))
from setred_package import setred_scratch, simulated_data,setred_scratch_mca
from utils.adspy_shared_utilities import plot_class_regions_for_classifier

In [3]:
X = np.load("../data/X.npy")
y = np.load("../data/y.npy")
X_unlabel = np.load("../data/X_unlabel.npy")
y_unlabel = np.load("../data/y_unlabel.npy")
X_ori = np.load("../data/X_ori.npy")
y_ori = np.load("../data/y_ori.npy")
X_test = np.load("../data/X_test.npy")
y_test = np.load("../data/y_test.npy")

In [4]:
df_X = pd.DataFrame(X, columns=[f'dim{i+1}' for i in range(X.shape[1])])
df_y = pd.DataFrame(y, columns=['target'])
df_X_unlabel = pd.DataFrame(X_unlabel, columns=[f'dim{i+1}' for i in range(X_unlabel.shape[1])])
df_y_unlabel = pd.DataFrame(y_unlabel, columns=['target_unlabel'])
df_X_ori = pd.DataFrame(X_ori, columns=[f'dim{i+1}' for i in range(X_ori.shape[1])])
df_y_ori = pd.DataFrame(y_ori, columns=['target_ori'])
df_X_test = pd.DataFrame(X_test, columns=[f'dim{i+1}' for i in range(X_test.shape[1])])
df_y_test = pd.DataFrame(y_test, columns=['target_test'])


In [5]:
df_X.index = pd.Index(range(4, 4 + len(df_X)))



In [6]:
# Features for the model 
mod_features = [ 'dim1', 'dim2', 'dim3']
graph_features = [ 'dim1', 'dim2', 'dim3', 'dim4', 'dim5']

In [7]:
# Filtering the labeled instances
X_val = df_X[y != -1]
y_val = y[y != -1]
# Print the shape of the validation and testing sets
print(f"Shape of X_val: {X_val.shape}, y_val: {y_val.shape}, X_test: {X_test.shape}, y_test: {y_test.shape}")
# Print the number of the classes in y_val and y_test
print(f"Frequencies of classes in y_val:\n {pd.Series(y_val).value_counts(normalize=False).sort_index()}")
print(f"Frequencies of classes in y_test:\n {pd.Series(y_test).value_counts(normalize=False).sort_index()}")

Shape of X_val: (136, 5), y_val: (136,), X_test: (3400, 5), y_test: (3400,)
Frequencies of classes in y_val:
 0    40
1    24
2    23
3    24
4    25
Name: count, dtype: int64
Frequencies of classes in y_test:
 0    680
1    680
2    680
3    680
4    680
Name: count, dtype: int64


Fitting a decision tree classifier

In [8]:
# Fitting a decision tree with hyperparameters tunning using cross validation
dt = DecisionTreeClassifier(random_state=42)
param_grid = {
    'max_depth': [None, 5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}


grid_search = GridSearchCV(dt, param_grid, cv=5)
print("The shape of X_val:", X_val.shape)
print("The shape of y_val:", y_val.shape)

grid_search.fit(X_val[mod_features], y_val)

# Best parameters
best_params = grid_search.best_params_
print("Best parameters:", best_params)

# Base estimator
base_estimator_dt = DecisionTreeClassifier(**best_params, random_state=42)

The shape of X_val: (136, 5)
The shape of y_val: (136,)
Best parameters: {'max_depth': None, 'min_samples_leaf': 2, 'min_samples_split': 2}


SETRED CLASSIFIER

In [9]:
import warnings
from sklearn.exceptions import UndefinedMetricWarning
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=UndefinedMetricWarning)

In [10]:
# Fitting the Setred classifier
ssl_clf_dt = setred_scratch_mca.Setred_scratch(base_estimator=base_estimator_dt,
                        graph_neighbors=20,
                        max_iterations=1,                             
                        htunning=True,
                        param_grid={
                                'max_depth': [None, 5, 10, 15],
                                'min_samples_split': [2, 5, 10],
                                'min_samples_leaf': [1, 2, 4, 5, 15]
                            },     
                        method= 'bernoulli'  ,                         
                        view =10)


In [11]:
ssl_clf_dt.fit(df_X, y, 
               mod_features=mod_features,
                 graph_features=graph_features[:2])

Setred_scratch(base_estimator=DecisionTreeClassifier(min_samples_leaf=2,
                                                     random_state=42),
               graph_neighbors=20, htunning=True, max_iterations=1,
               param_grid={'max_depth': [None, 5, 10, 15],
                           'min_samples_leaf': [1, 2, 4, 5, 15],
                           'min_samples_split': [2, 5, 10]},
               view=10)

In [12]:
ssl_clf_dt.X_label_

,dim1,dim2,dim3
4,-0.243005,1.434787,0.188725
5,1.505145,0.289987,-0.272472
6,0.292723,-0.990564,-1.229469
7,1.337928,-0.147603,-1.126702
8,-0.686311,1.410565,-0.493412
...,...,...,...
6883,-0.753838,-0.214951,-0.230058
9438,0.066802,0.133321,-0.385604
13477,-0.237904,-0.247322,-0.928934
7287,-0.328884,0.722561,-0.296102


In [16]:
136*2


272